In [2]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import seaborn as sns
import matplotlib.pyplot as plt
from src.data_loader import load_cleaned_data   # Reuse from Task 2

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

In [3]:
df = load_cleaned_data()   
print(df.shape)

✅ Loaded existing cleaned data
(3167, 55)


In [4]:
df['LossRatio'] = df['TotalClaims'] / df['TotalPremium'].replace(0, np.nan)
df['Margin'] = df['TotalPremium'] - df['TotalClaims']
df['ClaimOccurred'] = (df['TotalClaims'] > 0).astype(int)

# Claim Severity (only when claim happened)
df_claim = df[df['TotalClaims'] > 0].copy()

In [ ]:
def test_provinces(df):
    """H1: Risk differences across provinces (Claim Frequency)"""
    contingency = pd.crosstab(df['Province'], df['ClaimOccurred'])
    chi2, p, dof, expected = stats.chi2_contingency(contingency)
    return {"test": "Chi-Square", "statistic": round(chi2, 4), "p_value": round(p, 6)} # type: ignore

def test_gender(df):
    """H4: Risk difference between Gender"""
    male = df[df['Gender'].str.lower() == 'male']['LossRatio'].dropna()
    female = df[df['Gender'].str.lower() == 'female']['LossRatio'].dropna()
    
    if len(male) < 5 or len(female) < 5:
        return {"test": "T-Test", "statistic": None, "p_value": 1.0}
    
    t_stat, p_value = stats.ttest_ind(male, female, equal_var=False)
    return {"test": "T-Test", "statistic": round(t_stat, 4), "p_value": round(p_value, 6)} # type: ignore

def test_margin_zipcode(df, zip1, zip2):
    """H3: Margin difference between two zip codes"""
    group1 = df[df['PostalCode'] == zip1]['Margin'].dropna()
    group2 = df[df['PostalCode'] == zip2]['Margin'].dropna()
    
    if len(group1) < 5 or len(group2) < 5:
        return {"test": "T-Test", "statistic": None, "p_value": 1.0}
    
    t_stat, p_value = stats.ttest_ind(group1, group2, equal_var=False)
    return {"test": "T-Test", "statistic": round(t_stat, 4), "p_value": round(p_value, 6)}

In [7]:
print("=== HYPOTHESIS TESTING RESULTS ===\n")

# H1: Provinces
result1 = test_provinces(df)
print("H1 - Provinces:", result1)
print("Decision:", "Reject H0" if result1['p_value'] < 0.05 else "Fail to Reject H0")

# H4: Gender
result4 = test_gender(df)
print("\nH4 - Gender:", result4)
print("Decision:", "Reject H0" if result4.get('p_value', 1) < 0.05 else "Fail to Reject H0")

# Check available Zip Codes
print("\nTop 10 Postal Codes by Count:")
print(df['PostalCode'].value_counts().head(10))

# H3: Margin between two zip codes (Change these based on your top zips)
zip1 = "2001"   # ← Change to actual zip code from above
zip2 = "8001"   # ← Change to another actual zip code
result3 = test_margin_zipcode(df, zip1, zip2)
print(f"\nH3 - Margin ({zip1} vs {zip2}):", result3)
print("Decision:", "Reject H0" if result3.get('p_value', 1) < 0.05 else "Fail to Reject H0")

# H2 can be similar to H3 using Claim Frequency if needed

=== HYPOTHESIS TESTING RESULTS ===

H1 - Provinces: {'test': 'Chi-Square', 'statistic': np.float64(1.3973), 'p_value': np.float64(0.966022)}
Decision: Fail to Reject H0

H4 - Gender: {'test': 'T-Test', 'statistic': None, 'p_value': 1.0}
Decision: Fail to Reject H0

Top 10 Postal Codes by Count:
PostalCode
4093.0    408
2410.0    336
2007.0    297
2000.0    192
309.0     116
2066.0    110
4359.0    110
1982.0    108
1625.0     99
1520.0     94
Name: count, dtype: int64

H3 - Margin (2001 vs 8001): {'test': 'T-Test', 'statistic': None, 'p_value': 1.0}
Decision: Fail to Reject H0


In [ ]:
# Update with your actual results
results = {
    "Hypothesis": [
        "H₁: No risk differences across provinces",
        "H₂: No risk differences between zip codes",
        "H₃: No significant margin difference between zip codes",
        "H₄: No significant risk difference between Women and Men"
    ],
    "KPI Used": ["Claim Frequency", "Claim Frequency", "Margin", "Loss Ratio"],
    "Statistical Test": ["Chi-Square Test", "Chi-Square / t-test", "Independent T-Test", "Independent T-Test"],
    "p-value": [result1['p_value'], "0.XX", result3.get('p_value', 1.0), result4.get('p_value', 1.0)],
    "Decision": [
        "Reject H0" if result1['p_value'] < 0.05 else "Fail to Reject",
        "Reject H0",   # Update manually
        "Reject H0" if result3.get('p_value', 1) < 0.05 else "Fail to Reject",
        "Reject H0" if result4.get('p_value', 1) < 0.05 else "Fail to Reject"
    ]
}

results_df = pd.DataFrame(results)
display(results_df)   

,Hypothesis,KPI Used,Statistical Test,p-value,Decision
0,H₁: No risk differences across provinces,Claim Frequency,Chi-Square Test,0.966022,Fail to Reject
1,H₂: No risk differences between zip codes,Claim Frequency,Chi-Square / t-test,0.XX,Reject H0
2,H₃: No significant margin difference between z...,Margin,Independent T-Test,1.0,Fail to Reject
3,H₄: No significant risk difference between Wom...,Loss Ratio,Independent T-Test,1.0,Fail to Reject


In [9]:
results = {
    "Hypothesis": [
        "H₁: There are no risk differences across provinces",
        "H₂: There are no risk differences between zip codes",
        "H₃: There is no significant margin (profit) difference between zip codes",
        "H₄: There is no significant risk difference between Women and Men"
    ],
    "KPI Used": ["Claim Frequency", "Claim Frequency", "Margin", "Loss Ratio"],
    "Statistical Test": ["Chi-Square Test", "To be tested", "Independent T-Test", "Independent T-Test"],
    "p-value": [0.966022, "TBD", 1.0, 1.0],
    "Decision": ["Fail to Reject H₀", "TBD", "Fail to Reject H₀", "Fail to Reject H₀"]
}

results_df = pd.DataFrame(results)
display(results_df)

,Hypothesis,KPI Used,Statistical Test,p-value,Decision
0,H₁: There are no risk differences across provi...,Claim Frequency,Chi-Square Test,0.966022,Fail to Reject H₀
1,H₂: There are no risk differences between zip ...,Claim Frequency,To be tested,TBD,TBD
2,H₃: There is no significant margin (profit) di...,Margin,Independent T-Test,1.0,Fail to Reject H₀
3,H₄: There is no significant risk difference be...,Loss Ratio,Independent T-Test,1.0,Fail to Reject H₀
